# Erasus — Multimodal Unlearning (ImageBind, Tesla T4)

[ImageBind](https://github.com/facebookresearch/ImageBind) (Meta, 2023) maps
**6 modalities** (image, text, audio, depth, thermal, IMU) into a shared
embedding space. We use **CLIP ViT-B/32** — the vision-language backbone that
underpins ImageBind's image+text alignment — to demonstrate genuine
**multimodal unlearning**: after surgical removal via Erasus, the model loses
the ability to classify *forget* classes both through its vision encoder
(classification accuracy) **and** through cross-modal text prompts (zero-shot
accuracy).

**Setup:**
- **Model**: CLIP ViT-B/32 (~151M params, `open_clip`) with a linear
  classifier head fine-tuned on CIFAR-10
- **Forget set**: 500 images from classes `airplane` and `automobile`
- **Retain set**: 1500 images from the remaining 8 classes
- **Unlearning**: Gradient ascent with 7 coreset selectors at 9 fraction levels
- **Metrics**: classification accuracy (linear head) *and* zero-shot accuracy
  (text-prompt similarity) — measured separately on forget/retain splits

**Runtime:** ~20–30 min on a free Colab T4 GPU.

In [ ]:
!pip install -q git+https://github.com/OnePunchMonk/erasus.git matplotlib torchvision
!pip install -q open_clip_torch
import torch
print(f"CUDA: {torch.cuda.is_available()}, Device: {torch.cuda.get_device_name(0)}")

In [ ]:
import os
import time
import json
import warnings
from copy import deepcopy

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import open_clip
import numpy as np

warnings.filterwarnings("ignore")

# ------------------------------------------------------------------ #
# CLIP backbone + linear classifier
# ------------------------------------------------------------------ #

CIFAR10_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]


class CLIPClassifier(nn.Module):
    """CLIP ViT-B/32 vision encoder with a trainable linear head.

    The CLIP backbone is frozen during initial feature extraction but
    the final projection + classifier are fine-tuned so that erasus
    selectors can compute meaningful per-sample gradients.
    """

    def __init__(self, clip_model, embed_dim: int = 512, n_classes: int = 10):
        super().__init__()
        self.clip = clip_model
        # Trainable layers — the last transformer block + projection + head
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Linear(256, n_classes),
        )

    def encode_image(self, x: torch.Tensor) -> torch.Tensor:
        """Return L2-normalised image embeddings."""
        feat = self.clip.encode_image(x)
        feat = F.normalize(feat.float(), dim=-1)
        return feat

    def forward(self, x, labels=None, **kwargs):
        feat = self.encode_image(x)
        logits = self.classifier(feat)
        if labels is not None:
            return type("Out", (), {
                "logits": logits,
                "loss": F.cross_entropy(logits, labels),
            })()
        return logits


# ------------------------------------------------------------------ #
# Load CLIP ViT-B/32 (pretrained on OpenAI WebImageText)
# ------------------------------------------------------------------ #

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="openai",
)
tokenizer = open_clip.get_tokenizer("ViT-B-32")

model = CLIPClassifier(clip_model, embed_dim=512, n_classes=10).to(device)
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable params (before freeze setup): {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Freeze everything except the last transformer block, projection, and
# classifier head — keeps VRAM under the T4 15 GB limit while giving
# selectors something non-trivial to differentiate.
for name, param in model.clip.named_parameters():
    param.requires_grad = False
# Unfreeze the last resblock of the vision transformer
for name, param in model.clip.visual.transformer.resblocks[-1].named_parameters():
    param.requires_grad = True
# Unfreeze projection
if model.clip.visual.proj is not None:
    model.clip.visual.proj.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params (after partial freeze): {trainable:,}")

# ------------------------------------------------------------------ #
# Data — CIFAR-10 resized to 224x224 for CLIP
# ------------------------------------------------------------------ #

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.4814, 0.4578, 0.4082), (0.2686, 0.2613, 0.2758)),
])

cifar10 = datasets.CIFAR10(
    root="/content/data", train=True, download=True, transform=transform,
)

forget_classes = {0, 1}  # airplane, automobile
forget_indices, retain_indices = [], []

for idx, (_, label) in enumerate(cifar10):
    if label in forget_classes and len(forget_indices) < 500:
        forget_indices.append(idx)
    elif label not in forget_classes and len(retain_indices) < 1500:
        retain_indices.append(idx)
    if len(forget_indices) >= 500 and len(retain_indices) >= 1500:
        break

print(f"Forget samples: {len(forget_indices)}, Retain samples: {len(retain_indices)}")

forget_set = Subset(cifar10, forget_indices)
retain_set = Subset(cifar10, retain_indices)

forget_loader = DataLoader(forget_set, batch_size=32, shuffle=True)
retain_loader = DataLoader(retain_set, batch_size=32, shuffle=True)

train_set = Subset(cifar10, forget_indices + retain_indices)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)

# ------------------------------------------------------------------ #
# Zero-shot accuracy helper (text-prompt based)
# ------------------------------------------------------------------ #

@torch.no_grad()
def zero_shot_accuracy(model, loader, device, class_names=CIFAR10_CLASSES):
    """Measure accuracy by comparing image embeddings to text embeddings
    of 'a photo of a {class}' prompts — this is the cross-modal metric."""
    prompts = [f"a photo of a {c}" for c in class_names]
    text_tokens = tokenizer(prompts).to(device)
    text_features = model.clip.encode_text(text_tokens)
    text_features = F.normalize(text_features.float(), dim=-1)

    model.eval()
    correct = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        img_features = model.encode_image(imgs)
        # Cosine similarity -> predicted class
        sims = img_features @ text_features.T
        preds = sims.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return correct / total if total > 0 else 0.0


def compute_accuracy(model, loader, device):
    """Classification accuracy via the linear head."""
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            if hasattr(logits, "logits"):
                logits = logits.logits
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total if total > 0 else 0.0


# ------------------------------------------------------------------ #
# Fine-tune on CIFAR-10 (classifier head + last resblock)
# ------------------------------------------------------------------ #

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=3e-4, weight_decay=1e-2,
)

print("\nFine-tuning for 10 epochs ...")
for epoch in range(1, 11):
    model.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs, labels=labels)
        out.loss.backward()
        optimizer.step()
        running_loss += out.loss.item()
    if epoch % 2 == 0 or epoch == 1:
        print(f"  Epoch {epoch:>2d} | loss {running_loss / len(train_loader):.4f}")

base_forget_acc = compute_accuracy(model, forget_loader, device)
base_retain_acc = compute_accuracy(model, retain_loader, device)
base_forget_zs = zero_shot_accuracy(model, forget_loader, device)
base_retain_zs = zero_shot_accuracy(model, retain_loader, device)

print(f"\nBase model (linear head)   — Forget: {base_forget_acc:.4f} | Retain: {base_retain_acc:.4f}")
print(f"Base model (zero-shot)     — Forget: {base_forget_zs:.4f} | Retain: {base_retain_zs:.4f}")

trained_state = deepcopy(model.state_dict())

In [ ]:
from erasus import ErasusUnlearner

# ------------------------------------------------------------------ #
# Coreset ablation
# ------------------------------------------------------------------ #

FRACTIONS = [0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
SELECTORS = ["influence", "gradient_norm", "el2n", "tracin", "herding", "kcenter", "random"]

results = {}  # {selector: {fraction: {...}}}

total_runs = len(SELECTORS) * len(FRACTIONS)
run_idx = 0

for selector_name in SELECTORS:
    results[selector_name] = {}
    for frac in FRACTIONS:
        run_idx += 1
        tag = f"[{run_idx}/{total_runs}]"

        # Fresh copy of the trained model
        m = CLIPClassifier(clip_model, embed_dim=512, n_classes=10).to(device)
        m.load_state_dict(deepcopy(trained_state))

        try:
            unlearner = ErasusUnlearner(
                model=m,
                strategy="gradient_ascent",
                selector=selector_name,
                selector_config={"fraction": frac},
            )
            t0 = time.time()
            unlearner.fit(
                forget_data=forget_loader,
                retain_data=retain_loader,
                epochs=5,
            )
            elapsed = time.time() - t0

            f_acc = compute_accuracy(m, forget_loader, device)
            r_acc = compute_accuracy(m, retain_loader, device)
            f_zs = zero_shot_accuracy(m, forget_loader, device)
            r_zs = zero_shot_accuracy(m, retain_loader, device)
        except Exception as exc:
            print(f"{tag}  {selector_name:>15s}  frac={frac:.2f}  -- ERROR: {exc}")
            f_acc = r_acc = f_zs = r_zs = float("nan")
            elapsed = 0.0

        results[selector_name][frac] = {
            "forget_acc": f_acc,
            "retain_acc": r_acc,
            "forget_zs": f_zs,
            "retain_zs": r_zs,
            "time_s": round(elapsed, 2),
        }
        print(
            f"{tag}  {selector_name:>15s}  frac={frac:.2f}  |  "
            f"forget={f_acc:.4f}  retain={r_acc:.4f}  "
            f"zs_forget={f_zs:.4f}  zs_retain={r_zs:.4f}  ({elapsed:.1f}s)"
        )

# ------------------------------------------------------------------ #
# Summary table
# ------------------------------------------------------------------ #

print("\n" + "=" * 110)
print(
    f"{'Selector':>15s}  {'Frac':>5s}  "
    f"{'Forget':>8s}  {'Retain':>8s}  "
    f"{'ZS-Forget':>10s}  {'ZS-Retain':>10s}  {'Time(s)':>8s}"
)
print("-" * 110)
for sel in SELECTORS:
    for frac in FRACTIONS:
        r = results[sel][frac]
        print(
            f"{sel:>15s}  {frac:>5.2f}  "
            f"{r['forget_acc']:>8.4f}  {r['retain_acc']:>8.4f}  "
            f"{r['forget_zs']:>10.4f}  {r['retain_zs']:>10.4f}  {r['time_s']:>8.2f}"
        )
    print()
print("=" * 110)

In [ ]:
import matplotlib.pyplot as plt

# ------------------------------------------------------------------ #
# Prepare arrays
# ------------------------------------------------------------------ #

fracs = np.array(FRACTIONS)

forget_curves = {}
retain_curves = {}
forget_zs_curves = {}
retain_zs_curves = {}
trade_curves = {}

for sel in SELECTORS:
    fa = np.array([results[sel][f]["forget_acc"] for f in FRACTIONS])
    ra = np.array([results[sel][f]["retain_acc"] for f in FRACTIONS])
    fz = np.array([results[sel][f]["forget_zs"] for f in FRACTIONS])
    rz = np.array([results[sel][f]["retain_zs"] for f in FRACTIONS])
    forget_curves[sel] = fa
    retain_curves[sel] = ra
    forget_zs_curves[sel] = fz
    retain_zs_curves[sel] = rz
    trade_curves[sel] = (1.0 - fa) * ra

# ------------------------------------------------------------------ #
# 2x3 panel figure — classification + zero-shot side by side
# ------------------------------------------------------------------ #

fig, axes = plt.subplots(2, 3, figsize=(20, 10))

for sel in SELECTORS:
    # Row 1: classification (linear head)
    axes[0, 0].plot(fracs, forget_curves[sel], marker="o", label=sel)
    axes[0, 1].plot(fracs, retain_curves[sel], marker="o", label=sel)
    axes[0, 2].plot(fracs, trade_curves[sel], marker="o", label=sel)
    # Row 2: zero-shot (text prompts)
    axes[1, 0].plot(fracs, forget_zs_curves[sel], marker="s", label=sel)
    axes[1, 1].plot(fracs, retain_zs_curves[sel], marker="s", label=sel)
    zs_trade = (1.0 - forget_zs_curves[sel]) * retain_zs_curves[sel]
    axes[1, 2].plot(fracs, zs_trade, marker="s", label=sel)

# Row 1 labels
axes[0, 0].set_title("Forget Accuracy (linear head)")
axes[0, 0].axhline(base_forget_acc, ls="--", color="gray", alpha=0.5, label="base")
axes[0, 1].set_title("Retain Accuracy (linear head)")
axes[0, 1].axhline(base_retain_acc, ls="--", color="gray", alpha=0.5, label="base")
axes[0, 2].set_title("Tradeoff (1-forget)*retain (linear)")

# Row 2 labels
axes[1, 0].set_title("Forget Accuracy (zero-shot)")
axes[1, 0].axhline(base_forget_zs, ls="--", color="gray", alpha=0.5, label="base")
axes[1, 1].set_title("Retain Accuracy (zero-shot)")
axes[1, 1].axhline(base_retain_zs, ls="--", color="gray", alpha=0.5, label="base")
axes[1, 2].set_title("Tradeoff (1-forget)*retain (zero-shot)")

for ax in axes.flat:
    ax.set_xlabel("Coreset Fraction")
    ax.legend(fontsize=6)
    ax.grid(True, alpha=0.3)

axes[0, 0].set_ylabel("Accuracy (linear head)")
axes[1, 0].set_ylabel("Accuracy (zero-shot)")

fig.suptitle(
    "Erasus Multimodal Unlearning — CLIP ViT-B/32 on CIFAR-10\n"
    "Classification (top) vs Zero-Shot Cross-Modal (bottom)",
    fontsize=14, y=1.02,
)
fig.tight_layout()
fig.savefig("/content/multimodal_ablation_6panel.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved /content/multimodal_ablation_6panel.png")

# ------------------------------------------------------------------ #
# Scatter: classification vs zero-shot forget accuracy at frac=0.2
# ------------------------------------------------------------------ #

fig2, ax2 = plt.subplots(figsize=(8, 6))
for sel in SELECTORS:
    cls_f = results[sel][0.2]["forget_acc"]
    zs_f = results[sel][0.2]["forget_zs"]
    ax2.scatter(cls_f, zs_f, s=100, zorder=3)
    ax2.annotate(sel, (cls_f, zs_f), textcoords="offset points",
                 xytext=(6, 6), fontsize=9)

ax2.axhline(base_forget_zs, ls="--", color="gray", alpha=0.4, label="base zero-shot")
ax2.axvline(base_forget_acc, ls="--", color="gray", alpha=0.4, label="base classification")
ax2.set_xlabel("Forget Accuracy (linear head)")
ax2.set_ylabel("Forget Accuracy (zero-shot)")
ax2.set_title("Cross-Modal Unlearning Effectiveness (frac=0.2)")
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

fig2.tight_layout()
fig2.savefig("/content/crossmodal_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved /content/crossmodal_scatter.png")

In [ ]:
# ------------------------------------------------------------------ #
# Save results JSON and download
# ------------------------------------------------------------------ #

output = {
    "experiment": "multimodal_clip_cifar10_coreset_ablation",
    "model": "CLIP ViT-B/32 + linear head",
    "backbone_params": "151M (ViT-B/32)",
    "dataset": "CIFAR-10",
    "forget_classes": ["airplane", "automobile"],
    "forget_samples": len(forget_indices),
    "retain_samples": len(retain_indices),
    "finetune_epochs": 10,
    "unlearn_epochs": 5,
    "strategy": "gradient_ascent",
    "base_forget_acc": base_forget_acc,
    "base_retain_acc": base_retain_acc,
    "base_forget_zs": base_forget_zs,
    "base_retain_zs": base_retain_zs,
    "fractions": FRACTIONS,
    "selectors": SELECTORS,
    "results": {},
}

for sel in SELECTORS:
    output["results"][sel] = {}
    for frac in FRACTIONS:
        r = results[sel][frac]
        output["results"][sel][str(frac)] = {
            "forget_acc": round(r["forget_acc"], 6) if not np.isnan(r["forget_acc"]) else None,
            "retain_acc": round(r["retain_acc"], 6) if not np.isnan(r["retain_acc"]) else None,
            "forget_zs": round(r["forget_zs"], 6) if not np.isnan(r["forget_zs"]) else None,
            "retain_zs": round(r["retain_zs"], 6) if not np.isnan(r["retain_zs"]) else None,
            "time_s": r["time_s"],
        }

json_path = "/content/multimodal_ablation_results.json"
with open(json_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Results saved to {json_path}")
print(f"\nKey findings:")
print(f"  Base forget acc (linear): {base_forget_acc:.4f}")
print(f"  Base forget acc (zero-shot): {base_forget_zs:.4f}")
print(f"  Base retain acc (linear): {base_retain_acc:.4f}")
print(f"  Base retain acc (zero-shot): {base_retain_zs:.4f}")

# Best tradeoff at frac=0.2
best_sel = max(
    SELECTORS,
    key=lambda s: (
        (1 - results[s][0.2]["forget_acc"]) * results[s][0.2]["retain_acc"]
        if not np.isnan(results[s][0.2]["forget_acc"]) else -1
    ),
)
br = results[best_sel][0.2]
print(f"\n  Best selector at frac=0.2: {best_sel}")
print(f"    forget={br['forget_acc']:.4f}  retain={br['retain_acc']:.4f}")
print(f"    zs_forget={br['forget_zs']:.4f}  zs_retain={br['retain_zs']:.4f}")

# Download files (Colab-specific)
try:
    from google.colab import files
    files.download(json_path)
    files.download("/content/multimodal_ablation_6panel.png")
    files.download("/content/crossmodal_scatter.png")
except ImportError:
    print("\nNot running on Colab -- files saved locally under /content/")